# 19 · Trechos não operacionais: estão fora? E quanto o resultado depende disso?

> **Regra da disciplina (aula 3):** especificação FOCUS → código gerado pelo assistente → executar → *gate* contra uma resposta já conhecida → aceitar.
> **Quando o gate falha, a restrição que faltou entra em `U`. O código não é remendado.**
>
> Cada passo abaixo tem três partes: a especificação FOCUS enviada ao assistente, o gate declarado **antes** do código (com a fonte da resposta conhecida e da tolerância) e o código que voltou, executado e gateado. O registro de uso de IA é exportado ao final (`registro_uso_ia.csv`).

**Pergunta do autor (16/11, 1h48).** A análise exclui os trechos não operacionais, como na dissertação? A proporção em relação à malha tem de ser calculada sobre a malha em operação.

**Resposta (declarada antes de rodar a sensibilidade).** Sim: desde o notebook 11 ficam fora as células cujo campo de sentido de tráfego da Declaração de Rede de 2025 é "Interrompido" ou "Interrompido/Desativado". Este notebook confere a contagem e repete o protocolo do notebook 18 (origem móvel, horizonte de 3 anos, oito conferências) em três universos: A, o atual; B, o atual sem as células de capacidade vinculada igual a zero; C, todas as células, inclusive as de tráfego interrompido.

**Gates.** 9.119 células interrompidas excluídas; 20.936 em operação; o universo A reproduz a AUC de 0,752 do notebook 18 (tolerância de 0,002).

In [1]:
# === Setup: caminhos, bibliotecas e as três ferramentas (gate, gate_between, log_ia) ===
import os, sys, json, re, unicodedata, datetime as dt
import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
if os.environ.get("ET261400_BASE"):
    BASE = os.environ["ET261400_BASE"]                # definido pelo notebook 00 (execução em lote) ou pela execução local
elif IN_COLAB and os.path.exists("/content/ET261400/data"):
    BASE = "/content/ET261400"                        # pacote já descompactado na sessão do Colab
elif IN_COLAB and os.path.exists("/content/et261400_colab.zip"):
    import zipfile
    zipfile.ZipFile("/content/et261400_colab.zip").extractall("/content/ET261400")   # pacote enviado pelo painel de arquivos
    BASE = "/content/ET261400"
elif IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    BASE = "/content/drive/MyDrive/ET261400"          # pasta do projeto no Drive (ajustar se for outra)
else:
    BASE = os.environ.get("ET261400_BASE", os.path.abspath(".."))   # execução local: raiz do repositório
RAW, PROC = f"{BASE}/data/raw", f"{BASE}/data/processed"
FIG, TAB = f"{BASE}/results/figures", f"{BASE}/results/tables"
for d in (PROC, FIG, TAB):
    os.makedirs(d, exist_ok=True)
print("BASE =", BASE)

_GATES = []          # todo gate executado fica registrado aqui
_LOG_IA = []         # registro de uso de IA: passo, ferramenta, especificação, resultado

def gate(name, got, expected, tol=0.0, source=""):
    """Compara um número calculado com um número que já se tinha antes do código.
    tol vem da fonte (meio passo de arredondamento, faixa publicada), nunca é alargada até passar."""
    if got is None:
        print(f"[ -- ] {name}: nada calculado ainda"); return False
    ok = abs(got - expected) <= tol
    _GATES.append(dict(gate=name, got=got, expected=expected, tol=tol, ok=ok, source=source))
    print(f"[{'PASS' if ok else 'FAIL'}] {name}: obtido {got}, esperado {expected} (tol +/-{tol}) {('| ' + source) if source else ''}")
    return ok

def gate_between(name, got, low, high, source=""):
    """O mesmo, quando a fonte publica uma faixa e não um número."""
    if got is None:
        print(f"[ -- ] {name}: nada calculado ainda"); return False
    ok = low <= got <= high
    _GATES.append(dict(gate=name, got=got, expected=f"[{low}, {high}]", tol=None, ok=ok, source=source))
    print(f"[{'PASS' if ok else 'FAIL'}] {name}: obtido {got}, faixa [{low}, {high}] {('| ' + source) if source else ''}")
    return ok

def log_ia(passo, ferramenta, especificacao, resultado, verificacao):
    """Registro de uso de IA exigido pela disciplina: o que foi pedido, a que ferramenta, o que voltou e como foi verificado."""
    _LOG_IA.append(dict(data=dt.date.today().isoformat(), passo=passo, ferramenta=ferramenta,
                        especificacao=especificacao.strip(), resultado=resultado, verificacao=verificacao))

def exportar_registros(prefixo):
    pd.DataFrame(_GATES).to_csv(f"{TAB}/{prefixo}_gates.csv", index=False)
    pd.DataFrame(_LOG_IA).to_csv(f"{TAB}/{prefixo}_registro_uso_ia.csv", index=False)
    n_ok = sum(g["ok"] for g in _GATES)
    print(f"gates: {n_ok}/{len(_GATES)} PASS | registros de IA: {len(_LOG_IA)} | exportados em {TAB}")

def norm_txt(s):
    """Maiúsculas sem acento, espaços simples: chave de junção para nomes de pátios e estações."""
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", s).upper().strip()


BASE = /home/claude


In [2]:
import warnings; warnings.filterwarnings("ignore")
from scipy.stats import rankdata
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
C0=pd.read_csv(f"{PROC}/11_celulas_1km.csv",low_memory=False); E=pd.read_csv(f"{PROC}/11_eventos_celulas.csv"); E['ano']=E['ano'].astype(int); EV0=E[E['via']]
NUM=["dmin_patio","dmin_amv","frac_zona_patio"]+[f"f16p{j}" for j in range(5)]+[f"f16a{j}" for j in range(5)]+["rmin_geo"]+[f"fr_lt{l}" for l in (300,400,500,600,800)]+["rampa_max_pct","carga_eixo_t","taxa_dorm","gab_vert_m","gab_horiz_m","cap_vinc","dias_op","raio_min_m","vma_carregado"]
CAT=["dormente","perfil_trilho","fixacao","sinalizacao","controle_circ","bitola","sentido","n_linhas"]
def auc(y,s):
    n1=y.sum(); n0=len(y)-n1; return float((rankdata(s)[y==1].sum()-n1*(n1+1)/2)/(n1*n0))
def roda(C, nome):
    C=C.sort_values(["ferrovia","linha","km_a"]).reset_index(drop=True).copy(); N=len(C); idx=pd.Series(np.arange(N),index=C["cel_id"]); EV=EV0[EV0.cel_id.isin(C.cel_id)]
    M=np.zeros((15,N))
    for k in range(15):
        s=EV[EV.ano==2011+k].groupby("cel_id").size(); M[k,idx[s.index].values]=s.values
    for c in NUM: C[c]=pd.to_numeric(C[c],errors="coerce")
    for c in CAT: C[c]=C[c].astype(str).str.strip().replace({"nan":"NA","":"NA"})
    X=C[NUM].copy(); med=X.median().fillna(0.0)
    for c in NUM: X[c+"_na"]=X[c].isna().astype(float); X[c]=X[c].fillna(med[c])
    X=StandardScaler().fit_transform(pd.concat([X,pd.get_dummies(C[CAT],dtype=float)],axis=1).values)
    cl=np.select([C.dmin_patio.fillna(1e9)<=300,C.dmin_patio.fillna(1e9)<=600,C.dmin_patio.fillna(1e9)<=1200,C.dmin_patio.fillna(1e9)<=1800],[4,3,2,1],0); rc=np.select([C.rmin_geo.fillna(1e6)<500,C.rmin_geo.fillna(1e6)<1200,C.rmin_geo.fillna(1e6)<2300],[3,2,1],0); B2=0.5*cl/4+0.5*rc/3
    F=pd.factorize(C.ferrovia)[0]; LN=C.len_km.values; DES=np.random.default_rng(42).random(N); R=[]
    for T in range(2015,2023):
        ycal=(M[:T-2010].sum(0)>0).astype(int); cc=M[T-2010:T-2010+3].sum(0); y=(cc>0).astype(int)
        S={"regra":B2,"dummy malha":pd.Series(ycal).groupby(F).transform("mean").values,"EAR calibrado":LogisticRegression(C=0.5,max_iter=3000).fit(X,ycal).predict_proba(X)[:,1]}
        for m,s in S.items():
            a,w=[],[]
            for f in np.unique(F):
                i=F==f
                if 5<=y[i].sum()<i.sum(): a.append(auc(y[i],s[i])); w.append(y[i].sum())
            o=np.lexsort((DES,-s)); cum=np.cumsum(LN[o])/LN.sum(); d=dict(universo=nome,metodo=m,T=T,auc=auc(y,s),auc_dentro=np.average(a,weights=w),atingidos=int(y.sum()))
            for p in (0.2,0.4,0.5): d[f"acerto_{int(p*100)}"]=cc[o[cum<=p]].sum()/cc.sum()
            R.append(d)
    R=pd.DataFrame(R); g=R.groupby(["universo","metodo"],sort=False)[["auc","auc_dentro","acerto_20","acerto_40","acerto_50","atingidos"]].mean().round(3); g["celulas"]=N; g["eventos_via"]=len(EV); return g
op=~C0["sentido"].astype(str).str.contains("Interromp"); cv=pd.to_numeric(C0["cap_vinc"],errors="coerce").fillna(0)
out=pd.concat([roda(C0[op],"A: em operação pela Declaração de Rede (análise atual)"),roda(C0[op&(cv>0)],"B: em operação e com capacidade vinculada maior que zero"),roda(C0,"C: todas as células, inclusive tráfego interrompido")])
display(out); out.to_csv(f"{TAB}/19_sensibilidade_operacao.csv")
n_int=int((~op).sum()); gate("células com tráfego interrompido excluídas", n_int, 9119, source="Declaração de Rede 2025, campo sentido"); gate("células em operação", int(op.sum()), 20936, source="notebook 11")
gate("universo A reproduz o notebook 18 (AUC do EAR calibrado)", round(float(out.loc[("A: em operação pela Declaração de Rede (análise atual)", "EAR calibrado"), "auc"]), 3), 0.752, tol=0.002, source="notebook 18")
log_ia("19 sensibilidade ao filtro de operação", "Claude", "FOCUS: mesmo protocolo do notebook 18 em três universos; nada mais muda", "19_sensibilidade_operacao.csv", "3 gates")
exportar_registros("19")

auc  \
universo                                           metodo                 
A: em operação pela Declaração de Rede (análise... regra          0.580   
                                                   dummy malha    0.688   
                                                   EAR calibrado  0.752   
B: em operação e com capacidade vinculada maior... regra          0.579   
                                                   dummy malha    0.666   
                                                   EAR calibrado  0.752   
C: todas as células, inclusive tráfego interrom... regra          0.592   
                                                   dummy malha    0.609   
                                                   EAR calibrado  0.807   

                                                                  auc_dentro  \
universo                                           metodo                      
A: em operação pela Declaração de Rede (análise... regra               0.577   
                                                   dummy malha         0.500   
                                                   EAR calibrado       0.665   
B: em operação e com capacidade vinculada maior... regra               0.573   
                                                   dummy malha         0.500   
                                                   EAR calibrado       0.659   
C: todas as células, inclusive tráfego interrom... regra               0.594   
                                                   dummy malha         0.500   
                                                   EAR calibrado       0.769   

                                                                  acerto_20  \
universo                                           metodo                     
A: em operação pela Declaração de Rede (análise... regra              0.263   
                                                   dummy malha        0.357   
                                                   EAR calibrado      0.537   
B: em operação e com capacidade vinculada maior... regra              0.252   
                                                   dummy malha        0.337   
                                                   EAR calibrado      0.546   
C: todas as células, inclusive tráfego interrom... regra              0.304   
                                                   dummy malha        0.256   
                                                   EAR calibrado      0.612   

                                                                  acerto_40  \
universo                                           metodo                     
A: em operação pela Declaração de Rede (análise... regra              0.507   
                                                   dummy malha        0.657   
                                                   EAR calibrado      0.759   
B: em operação e com capacidade vinculada maior... regra              0.497   
                                                   dummy malha        0.576   
                                                   EAR calibrado      0.756   
C: todas as células, inclusive tráfego interrom... regra              0.536   
                                                   dummy malha        0.553   
                                                   EAR calibrado      0.876   

                                                                  acerto_50  \
universo                                           metodo                     
A: em operação pela Declaração de Rede (análise... regra              0.627   
                                                   dummy malha        0.809   
                                                   EAR calibrado      0.858   
B: em operação e com capacidade vinculada maior... regra              0.611   
                                                   dummy malha        0.730   
                                                   EAR calibrado    

[PASS] células com tráfego interrompido excluídas: obtido 9119, esperado 9119 (tol +/-0.0) | Declaração de Rede 2025, campo sentido
[PASS] células em operação: obtido 20936, esperado 20936 (tol +/-0.0) | notebook 11
[PASS] universo A reproduz o notebook 18 (AUC do EAR calibrado): obtido 0.752, esperado 0.752 (tol +/-0.002) | notebook 18
gates: 3/3 PASS | registros de IA: 1 | exportados em /home/claude/results/tables


---
## Leitura

Manter os trechos sem operação aumenta a AUC porque o modelo aprende que linha parada não tem descarrilamento, o que não é informação sobre a via. Por isso eles ficam fora. A exclusão segue o retrato de 2025: trechos interrompidos hoje podem ter operado em anos anteriores, e os descarrilamentos por via ocorridos neles saem junto.

**Declaração de uso de IA (regra da disciplina, três partes).** Ferramentas: Claude (Anthropic), usado pelo autor como assistente de programação a partir das especificações FOCUS registradas neste notebook, e Gemini do Colab quando indicado. Parte do trabalho: geração do código de cada passo e revisão do texto das células. Responsável pelo conteúdo: o autor, que definiu o problema, as fontes, as especificações, os gates e a verificação, e executou o notebook do zero antes de reportar qualquer número.